In [ ]:
import pandas as pd
from utils.misc import cols_to_front
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import TruncatedSVD
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import os
from sklearn.feature_extraction.text import TfidfTransformer
from utils.misc import cols_to_front

# Ensure output dir exists
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

REDUCE_DIM = False
SHOW_PLOT = False
n_dim = 100


In [ ]:
# Import data
merged_df = pd.read_csv('data/merged_ai_descriptors_clean.csv')
df1 = pd.read_csv('data/merged_ai_descriptors_dummies_filtered.csv')

In [ ]:
# Get column names
desc = [
    c for c in df1.columns
    if c != "Nom scientifique" and "cat_" not in c and 'category' not in c and 'db' not in c
]
desc_og = desc
cat_cols = [c for c in df1.columns if "cat_" in c]
name = "Nom scientifique"

In [ ]:
merged_df['category'] = merged_df['sub_category'].copy()

In [ ]:
df1 = df1.merge(
    merged_df[["Nom scientifique", "category", "db"]].drop_duplicates("Nom scientifique"),
    on="Nom scientifique",
    how="left",
    validate="many_to_one"  # optional but recommended
)

df1 = cols_to_front(df1, ['Nom scientifique', 'category'])

df1_og = df1.copy(deep=True)
df1.head()

# Overview

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count number of columns > 0 for each row
row_nonzero_counts = df1[desc].sum(axis=1)

if SHOW_PLOT:
    # Plot histogram
    plt.figure(figsize=(10, 6))
    sns.histplot(row_nonzero_counts, bins=20)
    plt.xlabel('Number descriptors', fontsize=12)
    plt.ylabel('Number of ingredients', fontsize=12)
    plt.title('Number of descriptors per ingredients', fontsize=14)
    plt.grid(False)

    save_path = os.path.join("output", "stats_nbr_descriptor_per_ing.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')

In [ ]:
# Number of ingredient where each ai descriptor is used
non_null_counts = df1[desc].sum()

if SHOW_PLOT:
    # Plot histogram
    plt.figure(figsize=(10, 6))
    sns.histplot(non_null_counts, bins=20)
    plt.xlabel('Number of ingredients', fontsize=12)
    plt.ylabel('Number of descriptors', fontsize=12)
    plt.title('Number of descriptor per ingredient', fontsize=14)
    plt.grid(False)

    save_path = os.path.join("output", "stats_nbr_of ingredients_per_descriptor.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
# Number of ingredient where each ai descriptor is used
non_null_counts.sort_values(ascending=False).head(20)

In [ ]:
# Number of ingredient where each ai descriptor is used
cat_flavors = {}
cat_flav10 = {}
df2 = df1[desc+["category"]].groupby('category').sum()

for cat in df2.index.unique():
    cat_flavors[cat] =  df2.loc[cat].sort_values(ascending=False)
    cat_flav10[cat] = cat_flavors[cat].head(10)


# Category definition

## Custom categories

In [ ]:
lexicon = {'plant': 'plant',
 'animal product': 'animal product',
 'fungus': 'fungus',
 'beverage': 'beverage',
 None: None,
 'vegetable': 'vegetable',
 'dish': 'dish',
 'vegetable-root': 'root',
 'fruit': 'fruit',
 'herb': 'herb',
 'additive': 'additive',
 'cereal': 'cereal',
 'flower': 'flower',
 'bakery': 'bakery',
 'beverage alcoholic': 'beverage alcoholic',
 'beverage caffeinated': 'beverage',
 'maize': 'cereal',
 'dairy': 'dairy',
 'essential oil': 'essential oil',
 'berry': 'berry',
 'seafood': 'seafood',
 'fish': 'seafood',
 'fruit-berry': 'berry',
 'fruit citrus': 'fruit citrus',
 'fruit essence': 'essential oil',
 'meat': 'animal product',
 'nut': 'nut',
 'seed': 'nut',
 'legume': 'legume',
 'plant derivative': 'plant derivative',
 'spice': 'spice',
 'cabbage': 'vegetable',
 'vegetable root': 'root',
 'vegetable fruit': 'vegetable fruit',
 'gourd': 'gourd',
 'vegetable stem': 'vegetable',
 'vegetable tuber': 'root'}

lexicon_og = {cat: cat for cat in df1['category'].dropna().unique()}  # make og dict

In [ ]:
df1['category'] = df1['category'].map(lexicon).fillna(df1['category'])

In [ ]:
cat_to_drop = ['animal product','dish', 'bakery', 'beverage alcoholic', 'dairy' ,'seafood' ]
df1 = df1[~df1['category'].isin(cat_to_drop)]

## FoodDB categories

In [ ]:
foodb = pd.read_csv("data/foodb_food_cat_list.csv")

foodb['name'] = foodb['name'].str.lower()
group_dict = foodb.set_index('name')['food_group'].to_dict()
subgroup_dict = foodb.set_index('name')['food_subgroup'].to_dict()

In [ ]:
# df1['Nom scientifique'] = df1['Nom scientifique'].str.lower()
# df1['category'] = df1['Nom scientifique'].map(group_dict).fillna(df1['category'])

# df1['category'] = df1['category'].str.lower()

In [ ]:
# cat = 'plant derivative'
# df1.loc[df1['category'] == cat]

In [ ]:
# df1 = df1.loc[df1['db'] != 'local']

In [ ]:
# df1 = df1.loc[df1['db'] == 'local']
# df1.loc[df1['category'] == 'fruit']

In [ ]:
# Keep only categories that appear at least 3 times
category_counts = df1['category'].value_counts()
df1 = df1[df1['category'].isin(category_counts[category_counts >= 3].index)]

In [ ]:
if SHOW_PLOT:
    # --- Build feature matrix (drop identifiers and one-hot category columns) ---
    df_features = df1[desc]
    df_features = df_features.drop(columns=[c for c in df_features.columns if "cat_" in c], errors='ignore')

    # Keep only numeric columns (t-SNE requires numbers)
    df_features = df_features.select_dtypes(include=[np.number])

    # Handle NaNs
    df_features = df_features.fillna(0.0)

    # Optional: scale features (recommended for t-SNE)
    X = StandardScaler().fit_transform(df_features.values)

    # --- Align categories from merged_df to df1 rows ---
    name_series = df1['Nom scientifique']
    categories = df1['category']

    # --- Fit t-SNE ---
    tsne = TSNE(
        n_components=2,
        perplexity=min(30, max(5, (len(df_features) - 1) // 3)),  # decent auto heuristic
        learning_rate='auto',
        init='pca',
        metric='euclidean',
        random_state=42,
    )
    tsne_embedding = tsne.fit_transform(X)

    # --- Prepare DataFrame for plotting ---
    tsne_df = pd.DataFrame(tsne_embedding, columns=['tSNE1', 'tSNE2'])
    tsne_df['category'] = categories.values

    # --- Plot ---
    plt.figure(figsize=(12, 8))
    sns.scatterplot(
        data=tsne_df,
        x='tSNE1',
        y='tSNE2',
        hue='category',
        s=50,
        alpha=0.8
    )
    plt.title('t-SNE projection of Ingredient Categories', fontsize=18)
    plt.xlabel('tSNE1', fontsize=14)
    plt.ylabel('tSNE2', fontsize=14)
    plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


In [ ]:
def jaccard_similarity_matrix(df_bin: pd.DataFrame) -> pd.DataFrame:
    """
    Compute Jaccard similarity for all pairs of rows in a binary DataFrame.
    """
    X = df_bin.to_numpy(dtype=int)
    n = X.shape[0]

    sim = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i, n):
            intersection = np.logical_and(X[i], X[j]).sum()
            union = np.logical_or(X[i], X[j]).sum()
            sim_val = intersection / union if union != 0 else 0
            sim[i, j] = sim_val
            sim[j, i] = sim_val

    return pd.DataFrame(sim, index=df_bin.index, columns=df_bin.index)

# Compute Jaccard similarity matrix
jaccard_sim = jaccard_similarity_matrix(df1[desc])

# Average similarity per category
jaccard_sim['category'] = df1['category'].values
jaccard_sim_mean = jaccard_sim.groupby('category').mean()
jaccard_sim_mean = jaccard_sim_mean.T
jaccard_sim_mean['category'] = df1['category'].values
jaccard_sim_mean = jaccard_sim_mean.groupby('category').mean()

# if SHOW_PLOT:
#     plt.figure(figsize=(10, 8))
#     sns.heatmap(jaccard_sim_mean, cmap="coolwarm", vmin=0, vmax = 0.5)
#     plt.title("Mean Jaccard Similarity of ingredients in each category")
#     plt.show()


In [ ]:
if SHOW_PLOT:
    # --- Cluster the category-level similarity matrix ---
    sns.clustermap(
        jaccard_sim_mean,
        cmap="coolwarm",
        vmin=0, vmax=0.5,
        method="ward",
        metric="euclidean",
        figsize=(10, 8)
    )

    plt.suptitle("Clustered Jaccard Similarity of Ingredient Categories", y=1.02)
    plt.show()

In [ ]:
cat = 'seafood'
df1.loc[df1['category'] == cat]
print(df1.loc[df1['category'] == cat]['Nom scientifique'].values)

In [ ]:
df1['category'].value_counts()

# PCA / dimensionality reduction

In [ ]:
# SVD
IDF = False # Seems to lead to worst fitting of dims by SVD
# --- Copy + select one-hot columns ---
df = df1.copy(deep=True)
X = df[desc]  # one-hot / sparse-ish features

# --- Truncated SVD (pick 100 components) ---
n_components = 100
svd = TruncatedSVD(n_components=n_components, random_state=42)
if IDF:    
    tfidf = TfidfTransformer(norm=None, use_idf=True, smooth_idf=True, sublinear_tf=False)
    X = tfidf.fit_transform(X)  # stays sparse
X_svd = svd.fit_transform(X)

if REDUCE_DIM == True:
    # --- Replace desc columns with SVD components in df ---
    svd_cols = [f"svd_{i+1:03d}" for i in range(n_components)]
    df.drop(columns=desc, inplace=True)
    df[svd_cols] = X_svd
    df1_dim_reduced = df.copy(deep=True)

In [ ]:
# --- "Explained variance ratio" equivalent for TruncatedSVD ---
explained_var = svd.explained_variance_ratio_  # length == n_components

# --- Scree plot (first 40) ---
k = min(40, len(explained_var))
plt.figure(figsize=(8, 5))
plt.plot(range(1, k + 1), explained_var[:k], marker="o", linestyle="--")
plt.title("Scree Plot (TruncatedSVD)")
plt.xlabel("Component")
plt.ylabel("Explained Variance Ratio")
plt.xticks(range(1, k + 1))
plt.grid(True, linestyle="--", alpha=0.4)

os.makedirs("output", exist_ok=True)
save_path = os.path.join("output", "scree_plot_svd.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

# --- Cumulative explained variance (first 40) ---
plt.figure(figsize=(8, 5))
plt.plot(range(1, k + 1), np.cumsum(explained_var[:k]), marker="o")
plt.title("Cumulative Explained Variance (TruncatedSVD)")
plt.xlabel("Component")
plt.ylabel("Cumulative Variance Ratio")
plt.xticks(range(1, k + 1))
plt.grid(True, linestyle="--", alpha=0.4)

save_path = os.path.join("output", "scree_plot_cum_svd.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# from sklearn.decomposition import PCA
# import matplotlib.pyplot as plt
# from sklearn.preprocessing import StandardScaler

# df = df1.copy(deep = True)

# # Center but don't scale to unit variance
# df = df[desc]
# cols = df.columns
# # scaler = StandardScaler(with_std=False)
# # df= scaler.fit_transform(df.drop(columns=['Nom scientifique']))


# pca = PCA()
# pca.fit(df)

# #  Explained variance ratio
# explained_var = pca.explained_variance_ratio_[:40]

# # Scree plot
# plt.figure(figsize=(8,5))
# plt.plot(range(1, len(explained_var)+1), explained_var, marker='o', linestyle='--')
# plt.title('Scree Plot')
# plt.xlabel('Principal Component')
# plt.ylabel('Explained Variance Ratio')
# plt.xticks(range(1, len(explained_var)+1))
# plt.grid(True, linestyle='--', alpha=0.4)

# # Save figure
# save_path = os.path.join("output", "scree_plot.png")
# plt.savefig(save_path, dpi=300, bbox_inches='tight')

# plt.show()

# # Optional: cumulative variance
# plt.figure(figsize=(8,5))
# plt.plot(range(1, len(explained_var)+1), explained_var.cumsum(), marker='o')
# plt.title('Cumulative Explained Variance')
# plt.xlabel('Principal Component')
# plt.ylabel('Cumulative Variance Ratio')
# plt.xticks(range(1, len(explained_var)+1))
# plt.grid(True, linestyle='--', alpha=0.4)

# # Save figure
# save_path = os.path.join("output", "scree_plot_cum.png")
# plt.savefig(save_path, dpi=300, bbox_inches='tight')

# plt.show()


In [ ]:
if 'pca' in globals():
    loadings = pd.DataFrame(pca.components_.T, columns=[f"PC{i+1}" for i in range(pca.n_components_)], index=cols)
if 'svd' in globals():
    loadings = pd.DataFrame(svd.components_.T, columns=[f"SVD{i+1}" for i in range(svd.n_components)], index=desc)

if 'loadings' in globals():
    top_features = {}

    for pc in loadings.columns[:20]:
        top10 = (
            loadings[pc]
            .nlargest(10)               # top 10
        )
        top_features[pc] = top10.index.tolist()

    # Print nicely
    for pc, features in top_features.items():
        print(f"{pc} top 10 features: {features}")

    # pc_theme = ["fruity", "earthy", "savory", "creamy", "Sour", "Spicy", "Toasted", "Oceanic", "Floral", "bright"]


In [ ]:
# saving
# ---- settings ----
top_n = 10
use_first_k_pcs = 20  # adjust as needed

# ---- pick model (PCA preferred if both exist) ----
if "pca" in globals() and hasattr(pca, "components_") and hasattr(pca, "explained_variance_ratio_"):
    model = pca
    prefix = "PC"
elif "svd" in globals() and hasattr(svd, "components_") and hasattr(svd, "explained_variance_ratio_"):
    model = svd
    prefix = "SVD"
else:
    raise NameError("Neither a fitted `pca` nor a fitted `svd` was found in globals().")

# ---- build loadings if not already defined ----
# Assumes you have `cols` (descriptor names). If not, fall back to `desc`.
if "loadings" not in globals() or loadings is None:
    feat_names = cols if "cols" in globals() else desc_og
    loadings = pd.DataFrame(
        model.components_.T,
        columns=[f"{prefix}{i+1}" for i in range(model.n_components)],
        index=feat_names,
    )

# ---- extract top features ----
pcs = loadings.columns[:use_first_k_pcs]
top_features = {pc: loadings[pc].nlargest(top_n).index.tolist() for pc in pcs}

top10_wide = pd.DataFrame.from_dict(top_features, orient="index")
top10_wide.index.name = prefix
top10_wide.columns = [f"Top{i}" for i in range(1, top_n + 1)]

# ---- add explained variance ----
explained_var = pd.Series(
    model.explained_variance_ratio_,
    index=[f"{prefix}{i+1}" for i in range(model.n_components)],
    name="explained_variance",
)

top10_wide["explained_variance (%)"] = explained_var.loc[top10_wide.index].values * 100
top10_wide = cols_to_front(top10_wide, ["explained_variance (%)"])

# ---- save ----
os.makedirs("output", exist_ok=True)
save_path = os.path.join("output", f"top{top_n}_features_wide_with_variance_{prefix}.xlsx")
top10_wide.to_excel(save_path)

print(f"✅ Saved with explained variance ({prefix}): {save_path}")

In [ ]:
# Plot data on new dims

# Manually choose which components to plot (1-indexed)
pc_a = 1
pc_b = 2

# --- Prepare data ---
X = df1[desc].copy()          # can be sparse or dense
categories = df1["category"]

# Fix index
i, j = pc_a - 1, pc_b - 1

# --- Use existing PCA/SVD if present, otherwise fit SVD ---
if "pca" in globals() and hasattr(pca, "components_"):
    model = pca
    pcs = model.transform(X)  # assumes X preprocessing matches how pca was fit
    prefix = "PC"
elif "svd" in globals() and hasattr(svd, "components_"):
    model = svd
    pcs = model.transform(X)
    prefix = "SVD"
else:
    n_components = max(100, pc_a, pc_b)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    pcs = svd.fit_transform(X)
    model = svd
    prefix = "SVD"

c1, c2 = pcs[:, i], pcs[:, j]
xvar = model.explained_variance_ratio_[i] * 100
yvar = model.explained_variance_ratio_[j] * 100

# --- Plot colored by category ---
plt.figure(figsize=(8, 7))
for cat in categories.unique():
    mask = (categories == cat).to_numpy()
    plt.scatter(c1[mask], c2[mask], label=cat, alpha=0.8)

plt.xlabel(f"{prefix}{pc_a} ({xvar:.2f}% var)")
plt.ylabel(f"{prefix}{pc_b} ({yvar:.2f}% var)")
plt.title(f"{prefix} Scatter Plot ({prefix}{pc_a} vs {prefix}{pc_b}) colored by category")

plt.axhline(0, color="grey", linewidth=0.8)
plt.axvline(0, color="grey", linewidth=0.8)
plt.legend(title="Category", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# # MCA - not ideal for one-hot encoded data with multiple labels like ours
# import prince

# df = df1.copy(deep = True)

# # Center but don't scale to unit variance
# df = df[desc]
# cols = df.columns

# # Ensure categorical dtype
# df_cat = df.astype("category")

# # Initialize MCA
# mca = prince.MCA(
#     n_components=2,
#     n_iter=5,
#     copy=True,
#     check_input=True,
#     engine="sklearn",
#     random_state=42
# )

# # Fit MCA
# mca = mca.fit(df_cat)

# # Get row coordinates (principal components)
# row_coords = mca.transform(df_cat)

# print(row_coords.head())

# # Get grouping variable from original df1
# group = df1['category'].astype("category")

# # Convert categories to integer codes
# group_codes = group.cat.codes
# unique_groups = group.cat.categories

# plt.figure()

# scatter = plt.scatter(
#     row_coords[0],
#     row_coords[1],
#     c=group_codes
# )

# plt.xlabel("PC1")
# plt.ylabel("PC2")
# plt.title("MCA - PC1 vs PC2")

# # Create legend manually
# handles = []
# for i, label in enumerate(unique_groups):
#     handles.append(
#         plt.Line2D(
#             [],
#             [],
#             marker='o',
#             linestyle='',
#             label=label
#         )
#     )

# plt.legend(
#     handles=handles,
#     title="category",
#     loc="center left",
#     bbox_to_anchor=(1, 0.5)
# )

# plt.tight_layout()
# plt.show()

In [ ]:
if REDUCE_DIM == True:
    desc = svd_cols  # update desc to point to new SVD columns
    df1 = df1_dim_reduced.copy(deep=True)

# Classification accuracy

In [ ]:
def jaccard_similarity_matrix(df_bin: pd.DataFrame) -> pd.DataFrame:
    """
    Compute Jaccard similarity for all pairs of rows in a binary DataFrame.
    """
    X = df_bin.to_numpy(dtype=int)
    n = X.shape[0]

    sim = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i, n):
            intersection = np.logical_and(X[i], X[j]).sum()
            union = np.logical_or(X[i], X[j]).sum()
            sim_val = intersection / union if union != 0 else 0
            sim[i, j] = sim_val
            sim[j, i] = sim_val

    return pd.DataFrame(sim, index=df_bin.index, columns=df_bin.index)

# Compute Jaccard similarity matrix
jaccard_sim = jaccard_similarity_matrix(df1[desc])

# Average similarity per category
jaccard_sim['category'] = df1['category'].values
jaccard_sim_mean = jaccard_sim.groupby('category').mean()
jaccard_sim_mean = jaccard_sim_mean.T
jaccard_sim_mean['category'] = df1['category'].values
jaccard_sim_mean = jaccard_sim_mean.groupby('category').mean()



# plt.figure(figsize=(10, 8))
# sns.heatmap(jaccard_sim_mean, cmap="coolwarm", vmin=0, vmax = 0.5)
# plt.title("Mean Jaccard Similarity of ingredinets in each category")
# save_path = os.path.join("output", "cat_jaccard.png")
# plt.savefig(save_path, dpi=300, bbox_inches='tight')

# plt.show()


In [ ]:
# --- Cluster the category-level similarity matrix ---
sns.clustermap(
    jaccard_sim_mean,
    cmap="coolwarm",
    vmin=0, vmax=0.5,
    method="ward",
    metric="euclidean",
    figsize=(10, 8)
)

plt.suptitle("Clustered Jaccard Similarity of Ingredient Categories", y=1.02)

save_path = os.path.join("output", "cat_clustered_jaccard.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
def get_top3_categories(sim_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each row in a similarity matrix, return the top 3 most similar
    categories (excluding self-similarity) and their similarity values.
    """
    df = sim_df.copy()

    # Sort columns for each row and take top 3
    top3_cols = df.apply(lambda row: row.nlargest(3).index.tolist(), axis=1)
    top3_vals = df.apply(lambda row: row.nlargest(3).values.tolist(), axis=1)

    # Build output DataFrame
    out = pd.DataFrame({
        "top1_category": top3_cols.str[0],
        "top1_jaccard": top3_vals.str[0],
        "top2_category": top3_cols.str[1],
        "top2_jaccard": top3_vals.str[1],
        "top3_category": top3_cols.str[2],
        "top3_jaccard": top3_vals.str[2],
    }, index=sim_df.index)

    return out


# ✅ Apply to your Jaccard similarity matrix
top3_jaccard = get_top3_categories(jaccard_sim_mean)

# top3_jaccard.to_excel("output/cat_jaccard_sim_top3.xlsx")


In [ ]:
import pandas as pd

def to_two_level_cols(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert columns like:
      top1_category, top1_jaccard, top2_category, top2_jaccard, top3_category, top3_jaccard
    into a MultiIndex:
      ('Top 1','category'), ('Top 1','jaccard similarity'), ...
    """
    rank_map = {"top1": "Top 1", "top2": "Top 2", "top3": "Top 3"}
    new_cols = []

    for c in df.columns:
        if "_" in c:
            rank, field = c.split("_", 1)
            top = rank_map.get(rank, rank.title())
            bottom = "jaccard similarity" if field.lower().startswith("jaccard") else field.lower()
            new_cols.append((top, bottom))
        else:
            # fallback if any extra columns
            new_cols.append(("Other", c))

    out = df.copy()
    out.columns = pd.MultiIndex.from_tuples(new_cols, names=["rank", "Reference Category"])
    return out

# ✅ Apply
top3_jaccard_mi = to_two_level_cols(top3_jaccard)

# Example access:
# top3_jaccard_mi[('Top 2', 'category')]
# top3_jaccard_mi[('Top 3', 'jaccard similarity')]
top3_jaccard_mi.to_excel("output/cat_jaccard_sim_top3.xlsx")

top3_jaccard_mi


In [ ]:
from sklearn.metrics.pairwise import cosine_distances
import pandas as pd

# desc = list of descriptor column names
# df1 = your main dataframe (columns: desc + ['db','category'])

# 1. Split into flavor_db and local
df_flavor = df1[df1["db"] == "flavor_db"].copy(deep=True)
df_local  = df1[df1["db"] != "flavor_db"].copy(deep=True)

# 2. Find categories that exist in both
shared_categories = set(df_flavor["category"]) & set(df_local["category"])

# 3. Prepare output list
results = []

# 4. Compute cosine distance for each shared category
for cat in shared_categories:
    v1 = df_flavor[df_flavor["category"] == cat][desc].mean().values
    v2 = df_local[df_local["category"] == cat][desc].mean().values
    
    dist = cosine_distances([v1], [v2])[0][0]
    
    results.append({
        "category": cat,
        "cosine_distance": dist
    })

# 5. Convert to DataFrame
distance_df = pd.DataFrame(results).sort_values("cosine_distance")
distance_df


In [ ]:
from sklearn.metrics.pairwise import cosine_distances
import pandas as pd

# Split two supercategories
df_flavor = df1[df1["db"] == "flavor_db"]
df_local  = df1[df1["db"] != "flavor_db"]

# Shared categories
shared_categories = set(df_flavor["category"]) & set(df_local["category"])

# Compute FlavorDB vs Local distance
between_results = []
for cat in shared_categories:
    v_flavor = df_flavor[df_flavor["category"] == cat][desc].mean().values
    v_local  = df_local[df_local["category"] == cat][desc].mean().values
    dist = cosine_distances([v_flavor], [v_local])[0][0]
    
    between_results.append({
        "category": cat,
        "between_distance": dist
    })

between_df = pd.DataFrame(between_results)

def intra_category_distances(df, desc):
    results = []
    for cat, group in df.groupby("category"):
        X = group[desc].values
        
        # Skip categories with <2 samples
        if len(X) < 2:
            continue
        
        # compute full distance matrix
        D = cosine_distances(X, X)
        
        # take upper triangle (excluding diag)
        triu = D[np.triu_indices_from(D, k=1)]
        results.append({
            "category": cat,
            "within_distance_mean": triu.mean(),
            "within_distance_std": triu.std(),
            "n_samples": len(X)
        })
    return pd.DataFrame(results)

within_flavor = intra_category_distances(df_flavor, desc)
within_local  = intra_category_distances(df_local, desc)

within_flavor["db"] = "flavor_db"
within_local["db"]  = "local"

within_df = pd.concat([within_flavor, within_local], ignore_index=True)

comparison = (
    between_df
    .merge(within_df[within_df["db"]=="flavor_db"][["category","within_distance_mean"]]
           .rename(columns={"within_distance_mean":"within_flavor_db"}),
           on="category", how="left")
    .merge(within_df[within_df["db"]=="local"][["category","within_distance_mean"]]
           .rename(columns={"within_distance_mean":"within_local"}),
           on="category", how="left")
)

comparison = comparison.sort_values("between_distance")
comparison


In [ ]:
# Compare shared categories between local and flavordb using jaccard

# Compute Jaccard similarity matrix
df2 = df1.loc[df1['db'] == 'local']
local_sim = jaccard_similarity_matrix(df2[desc])

# Average similarity per category
local_sim['category'] = df2['category'].values
local_sim_mean = local_sim.groupby('category').mean()
local_sim_mean = local_sim_mean.T
local_sim_mean['category'] = df2['category'].values
local_sim_mean = local_sim_mean.groupby('category').mean()
local_sim_mean

# Compute Jaccard similarity matrix
df2 = df1.loc[df1['db'] != 'local']
flavor_sim = jaccard_similarity_matrix(df2[desc])

# Average similarity per category
flavor_sim['category'] = df2['category'].values
flavor_sim_mean = flavor_sim.groupby('category').mean()
flavor_sim_mean = flavor_sim_mean.T
flavor_sim_mean['category'] = df2['category'].values
flavor_sim_mean = flavor_sim_mean.groupby('category').mean()
flavor_sim_mean

In [ ]:
# Compute similarity between ingredients within flavord vs local - should do by catgories though
# Compute Jaccard similarity matrix
jaccard_sim = jaccard_similarity_matrix(df1[desc])

# Average similarity per db
jaccard_sim['db'] = df1['db'].values
# jaccard_sim['category'] = df1['category'].values
jaccard_sim_mean = jaccard_sim.groupby('db').mean()
jaccard_sim_mean = jaccard_sim_mean.T
jaccard_sim_mean['db'] = df1['db'].values
jaccard_sim_mean = jaccard_sim_mean.groupby('db').mean()

jaccard_sim_mean


In [ ]:
# Compute similarity between ingredients within flavord vs local  
# Could maybe do a heatmap or cluster map of that 

# Compute Jaccard similarity matrix
jaccard_sim = jaccard_similarity_matrix(df1[desc])

# Average similarity per db
jaccard_sim['db'] = df1['db'].values
jaccard_sim['category'] = df1['category'].values

shared_cat = jaccard_sim.loc[jaccard_sim['db'] == 'local', 'category'].unique()

rows =[]
for cat in shared_cat:
    df2 = jaccard_sim.loc[jaccard_sim['category'] == cat]
    df2 = df2.drop(columns= 'category')
    idx = df2.index.to_list() + ['db']

    df2= df2[idx]
    jaccard_sim_mean = df2.groupby('db').mean()
    jaccard_sim_mean = jaccard_sim_mean.T
    jaccard_sim_mean['db'] = df2['db'].values
    jaccard_sim_mean = jaccard_sim_mean.groupby('db').mean()

    # get counts
    flavor_count = df2['db'].value_counts()['flavor_db']
    local_count = df2['db'].value_counts()['local']

    # Extract group diff
    flavor_local = jaccard_sim_mean.loc[jaccard_sim_mean.index == 'flavor_db']['local'].values[0]
    flavor_flavor = jaccard_sim_mean.loc[jaccard_sim_mean.index == 'flavor_db']['flavor_db'].values[0]
    local_local = jaccard_sim_mean.loc[jaccard_sim_mean.index == 'local']['local'].values[0]

    rows.append({
        "category": cat,
        "flavor_vs_local": flavor_local,
        "flavor_vs_flavor": flavor_flavor,
        "local_vs_local": local_local,
        'flavor_count' : flavor_count,
        'local_count' : local_count
    })

# Final comparison table
comparison_df = pd.DataFrame(rows)

comparison_df = comparison_df[comparison_df["local_count"] >= 3].copy()

comparison_df.to_excel(f"output/local_flavordb_jaccard_distance.xlsx", index=False)
comparison_df



# Umap

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns

local = merged_df[merged_df['db'] == 'local']
local_names = local['Nom scientifique'].to_list()

df1_local = df1[df1['Nom scientifique'].isin(local_names)]
df1_local

# Separate features and the category for coloring
df_features = df1[desc]
# df_features = df1_local.drop(columns=['Nom scientifique'])
df_features = df_features.drop(columns=[col for col in df_features.columns if "cat_" in col])
categories = merged_df['category'] # Use the category from the merged_df before dropping columns

# Apply UMAP
reducer = umap.UMAP(random_state=42)
umap_embedding = reducer.fit_transform(df_features)

# Create a DataFrame for plotting
umap_df = pd.DataFrame(umap_embedding, columns=['UMAP1', 'UMAP2'])
umap_df['category'] = categories

# Plot the UMAP embedding
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x='UMAP1',
    y='UMAP2',
    hue='category',
    data=umap_df,
    s=50,
    alpha=0.7
)
plt.title('UMAP projection of Ingredient Descriptors', fontsize=18)
plt.xlabel('UMAP1', fontsize=14)
plt.ylabel('UMAP2', fontsize=14)
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# --- Build feature matrix (drop identifiers and one-hot category columns) ---
df_features = df1[desc]
df_features = df_features.drop(columns=[c for c in df_features.columns if "cat_" in c], errors='ignore')

# Keep only numeric columns (t-SNE requires numbers)
df_features = df_features.select_dtypes(include=[np.number])

# Handle NaNs
df_features = df_features.fillna(0.0)

# Optional: scale features (recommended for t-SNE)
X = StandardScaler().fit_transform(df_features.values)

# --- Align categories from merged_df to df1 rows ---
name_series = df1['Nom scientifique']
categories = df1['category']

# --- Fit t-SNE ---
tsne = TSNE(
    n_components=2,
    perplexity=min(30, max(5, (len(df_features) - 1) // 3)),  # decent auto heuristic
    learning_rate='auto',
    init='pca',
    metric='euclidean',
    random_state=42,
)
tsne_embedding = tsne.fit_transform(X)

# --- Prepare DataFrame for plotting ---
tsne_df = pd.DataFrame(tsne_embedding, columns=['tSNE1', 'tSNE2'])
tsne_df['category'] = categories.values

# --- Plot ---
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=tsne_df,
    x='tSNE1',
    y='tSNE2',
    hue='category',
    s=50,
    alpha=0.8
)
plt.title('t-SNE projection of Ingredient Descriptors', fontsize=18)
plt.xlabel('tSNE1', fontsize=14)
plt.ylabel('tSNE2', fontsize=14)
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

save_path = os.path.join("output", "cat_tsne.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# --- Build feature matrix (drop identifiers and one-hot db columns) ---
df_features = df1[desc]
df_features = df_features.drop(columns=[c for c in df_features.columns if "cat_" in c], errors='ignore')

# Keep only numeric columns (t-SNE requires numbers)
df_features = df_features.select_dtypes(include=[np.number])

# Handle NaNs
df_features = df_features.fillna(0.0)

# Optional: scale features (recommended for t-SNE)
X = StandardScaler().fit_transform(df_features.values)

# --- Align categories from merged_df to df1 rows ---
name_series = df1['Nom scientifique']
categories = df1['db']

# --- Fit t-SNE ---
tsne = TSNE(
    n_components=2,
    perplexity=min(30, max(5, (len(df_features) - 1) // 3)),  # decent auto heuristic
    learning_rate='auto',
    init='pca',
    metric='euclidean',
    random_state=42,
)
tsne_embedding = tsne.fit_transform(X)

# --- Prepare DataFrame for plotting ---
tsne_df = pd.DataFrame(tsne_embedding, columns=['tSNE1', 'tSNE2'])
tsne_df['db'] = categories.values

# --- Plot ---
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=tsne_df,
    x='tSNE1',
    y='tSNE2',
    hue='db',
    s=50,
    alpha=0.8
)
plt.title('t-SNE projection of Ingredient Descriptors', fontsize=18)
plt.xlabel('tSNE1', fontsize=14)
plt.ylabel('tSNE2', fontsize=14)
plt.legend(title='db', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

save_path = os.path.join("output", "db_tsne.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Separate features and the category for coloring
df_features = df1[desc]
databases = merged_df['db'] # Use the 'db' column from the merged_df for coloring

# Apply UMAP
reducer = umap.UMAP(random_state=42)
umap_embedding = reducer.fit_transform(df_features)

# Create a DataFrame for plotting
umap_df = pd.DataFrame(umap_embedding, columns=['UMAP1', 'UMAP2'])
umap_df['database'] = databases

# Plot the UMAP embedding
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x='UMAP1',
    y='UMAP2',
    hue='database',
    data=umap_df,
    s=50,
    alpha=0.7
)
plt.title('UMAP projection of Ingredient Descriptors by Database', fontsize=18)
plt.xlabel('UMAP1', fontsize=14)
plt.ylabel('UMAP2', fontsize=14)
plt.legend(title='Database', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# KNN

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# --- Features and labels ---
X = df1[desc].values
y = df1['category']

# --- Split train/test ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- Scale features (important for distance-based models) ---
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# --- Test different k values ---
k_values = range(1, 21)
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    accuracies.append(accuracy_score(y_test, y_pred))

# --- Plot accuracy vs k ---
plt.figure(figsize=(8, 5))
plt.plot(k_values, accuracies, marker='o')
plt.title("KNN Classification Accuracy vs k")
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Accuracy")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# --- Prepare feature matrix ---
X = df1[desc].values

# --- Standardize features (important for clustering) ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Try several k values ---
k_values = [3, 5, 7, 8, 10]

# --- Run t-SNE once (expensive) ---
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_embedded = tsne.fit_transform(X_scaled)

# --- Plot clusters for each k ---
fig, axes = plt.subplots(1, len(k_values), figsize=(4*len(k_values), 4))

for ax, k in zip(axes, k_values):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(X_scaled)
    
    sc = ax.scatter(
        X_embedded[:, 0], X_embedded[:, 1],
        c=labels, cmap='tab10', s=30, alpha=0.8
    )
    ax.set_title(f"K-Means (k={k})")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# --- Prepare data ---
X = df1[desc].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Run K-Means clustering with k=7 ---
k = 7
kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
labels = kmeans.fit_predict(X_scaled)

# --- Add cluster labels to df1 ---
df1['cluster'] = labels

# --- Compute cluster centroids in original feature space ---
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=desc
)

# --- For each cluster, find top descriptors (highest centroid values) ---
top_desc_per_cluster = {}

for i in range(k):
    centroid = centroids.iloc[i]
    top_features = centroid.sort_values(ascending=False).head(10).index.tolist()
    top_desc_per_cluster[i] = top_features

# --- Display results ---
for cluster_id, top_feats in top_desc_per_cluster.items():
    print(f"\n🧭 Cluster {cluster_id}: Top descriptors")
    print(", ".join(top_feats))


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# --- params ---
k = 7
top_n = 5                       # how many nearest ingredients to list per cluster
feature_cols = desc             # your descriptor columns

# --- scale & cluster (if not already done) ---
X = df1[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
labels = kmeans.fit_predict(X_scaled)
centers = kmeans.cluster_centers_

df1['cluster'] = labels

# --- choose a display name column if available ---
name_col = 'name' if 'name' in df1.columns else ('Nom scientifique' if 'Nom scientifique' in df1.columns else None)
def get_label(idx):
    if name_col is not None:
        return df1.iloc[idx][name_col]
    return df1.index[idx]

# --- compute distances to own cluster centroid & pick top_n nearest ---
nearest_per_cluster = {}
nearest_rows = []  # for a tidy DataFrame output

for c in range(k):
    idx_c = np.where(labels == c)[0]
    if len(idx_c) == 0:
        nearest_per_cluster[c] = []
        continue

    Xc = X_scaled[idx_c]
    centroid = centers[c]

    # Euclidean distances in the scaled space
    dists = np.linalg.norm(Xc - centroid, axis=1)
    order = np.argsort(dists)[:min(top_n, len(idx_c))]

    closest_indices = idx_c[order]
    closest_names = [get_label(i) for i in closest_indices]
    closest_dists = dists[order]

    nearest_per_cluster[c] = list(zip(closest_names, closest_indices, closest_dists))

    # collect rows for a tidy table
    for name, idx_global, dist in zip(closest_names, closest_indices, closest_dists):
        nearest_rows.append({
            "cluster": c,
            "ingredient": name,
            "row_index": idx_global,
            "distance_to_centroid": float(dist)
        })

# --- dict result ---
# nearest_per_cluster: { cluster_id: [(ingredient_name, row_index, distance), ...], ... }

# --- tidy DataFrame result (sorted) ---
nearest_df = pd.DataFrame(nearest_rows).sort_values(["cluster", "distance_to_centroid"])
print(nearest_df)


In [ ]:
# Compare different categories in distance - not very useful

# def weighted_jaccard_similarity(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Compute the Weighted Jaccard similarity matrix for a numeric dataframe.
#     Rows = items, Columns = features.
#     """
#     X = df.to_numpy(dtype=float)
#     n = X.shape[0]
    
#     sim = np.zeros((n, n))
    
#     for i in range(n):
#         for j in range(i, n):
#             min_sum = np.minimum(X[i], X[j]).sum()
#             max_sum = np.maximum(X[i], X[j]).sum()
#             sim_val = min_sum / max_sum if max_sum != 0 else 0
#             sim[i, j] = sim_val
#             sim[j, i] = sim_val  # symmetric

#     return pd.DataFrame(sim, index=df.index, columns=df.index)

# # --- Usage ---
# # Remove non-numeric columns if needed
# df2_numeric = df2.select_dtypes(include=[np.number])

# wj_sim_matrix = weighted_jaccard_similarity(df2_numeric)
# wj_sim_matrix

# import seaborn as sns
# import matplotlib.pyplot as plt

# plt.figure(figsize=(10, 8))
# sns.heatmap(wj_sim_matrix, cmap="viridis", annot=False)
# plt.title("Weighted Jaccard Similarity Matrix")
# plt.show()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def cosine_similarity_matrix(df_num: pd.DataFrame) -> pd.DataFrame:
    """
    Compute cosine similarity for all pairs of rows in a numeric DataFrame.
    """
    X = df_num.to_numpy(dtype=float)
    n = X.shape[0]
    sim = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i, n):
            dot = np.dot(X[i], X[j])
            norm_i = np.linalg.norm(X[i])
            norm_j = np.linalg.norm(X[j])
            sim_val = dot / (norm_i * norm_j) if norm_i != 0 and norm_j != 0 else 0
            sim[i, j] = sim_val
            sim[j, i] = sim_val

    return pd.DataFrame(sim, index=df_num.index, columns=df_num.index)


# --- Compute cosine similarity matrix ---
cosine_sim = cosine_similarity_matrix(df1[desc])

# --- Average similarity per category ---
cosine_sim['category'] = df1['category'].values
cosine_sim_mean = cosine_sim.groupby('category').mean()
cosine_sim_mean = cosine_sim_mean.T
cosine_sim_mean['category'] = df1['category'].values
cosine_sim_mean = cosine_sim_mean.groupby('category').mean()

# --- Plot ---
plt.figure(figsize=(10, 8))
sns.heatmap(cosine_sim_mean, cmap="coolwarm", vmin=0, vmax=1)
plt.title("Mean Cosine Similarity of Ingredients in Each Category")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

def cosine_similarity_matrix(df_bin: pd.DataFrame) -> pd.DataFrame:
    """
    Compute cosine similarity for all pairs of rows in a DataFrame.
    """
    X = df_bin.to_numpy(dtype=float)
    sim = cosine_similarity(X)
    return pd.DataFrame(sim, index=df_bin.index, columns=df_bin.index)


# Compute Cosine similarity matrix
cosine_sim = cosine_similarity_matrix(df1[desc])

# Average similarity per category
cosine_sim['category'] = df1['category'].values
cosine_sim_mean = cosine_sim.groupby('category').mean()
cosine_sim_mean = cosine_sim_mean.T
cosine_sim_mean['category'] = df1['category'].values
cosine_sim_mean = cosine_sim_mean.groupby('category').mean()

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cosine_sim_mean, cmap="coolwarm", vmin=0, vmax=1)
plt.title("Mean Cosine Similarity of ingredients in each category")
plt.show()


In [ ]:
# --- Cluster the category-level similarity matrix ---
sns.clustermap(
    cosine_sim_mean,
    cmap="coolwarm",
    vmin=0, vmax=0.5,
    method="ward",
    metric="euclidean",
    figsize=(10, 8)
)

plt.suptitle("Clustered Jaccard Similarity of Ingredient Categories", y=1.02)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from networkx.algorithms import community

def plot_similarity_network(sim_df: pd.DataFrame, 
                            threshold: float = 0.15,
                            layout: str = "spring",
                            figsize=(12, 10)):
    """
    Build and plot a network graph from a category–category similarity matrix.

    threshold : only edges with similarity >= threshold are drawn
    layout : "spring", "kamada", "circular"
    """
    df = sim_df.copy()

    # Remove self-similarity
    np.fill_diagonal(df.values, 0)

    # Create graph
    G = nx.Graph()

    # Add nodes
    for cat in df.index:
        G.add_node(cat)

    # Add edges above threshold
    for i, cat1 in enumerate(df.index):
        for j, cat2 in enumerate(df.columns):
            if j <= i:
                continue  # avoid double counting
            weight = df.iloc[i, j]
            if weight >= threshold:
                G.add_edge(cat1, cat2, weight=weight)

    # Compute layout
    if layout == "spring":
        pos = nx.spring_layout(G, seed=42, k=0.4)
    elif layout == "kamada":
        pos = nx.kamada_kawai_layout(G)
    elif layout == "circular":
        pos = nx.circular_layout(G)
    else:
        pos = nx.spring_layout(G, seed=42)

    # Detect communities for node colors
    communities = list(community.greedy_modularity_communities(G))
    community_map = {}
    for c_id, comm in enumerate(communities):
        for node in comm:
            community_map[node] = c_id

    # Node colors based on community
    node_colors = [community_map[n] for n in G.nodes()]

    # Edge weights for thickness
    edges = G.edges(data=True)
    edge_weights = [d['weight'] * 4 for _,_,d in edges]  # scale for visibility

    # Plot
    plt.figure(figsize=figsize)

    nx.draw_networkx_nodes(
        G, pos,
        node_size=900,
        node_color=node_colors,
        cmap="tab20"
    )

    nx.draw_networkx_edges(
        G, pos,
        width=edge_weights,
        alpha=0.6
    )

    nx.draw_networkx_labels(
        G, pos,
        font_size=10,
        font_weight="bold"
    )

    plt.title(f"Category Similarity Network (threshold={threshold})")
    plt.axis("off")
    plt.show()

    return G

# df_to_graph = jaccard_sim.loc[:, jaccard_sim.columns != 'category'].copy(deep = True)
df_to_graph = jaccard_sim.loc[:, jaccard_sim.columns != 'category'].copy(deep = True)

to_drop = [954, 1030, 385, 1134]

df_to_graph  = df_to_graph .drop(index=to_drop, columns=to_drop)

# ✅ Build your network graph from Jaccard similarity
G_jaccard = plot_similarity_network(df_to_graph, threshold=0.15)
